In [ ]:
# Thư viện
import os
import re
import sys
import subprocess
import warnings
import unicodedata
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from sklearn.model_selection import (train_test_split,KFold,cross_validate,RandomizedSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score,mean_absolute_error,mean_squared_error,r2_score,mean_absolute_percentage_error)
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# Style và tùy chọn hiển thị
sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.dpi": 110,"figure.facecolor": "white",
    "axes.facecolor": "white","axes.spines.top": False,
    "axes.spines.right": False,"font.size": 11})
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", "{:.4f}".format)

ACCENT   = "#2563EB"   # Linear Regression
ORANGE   = "#EA580C"   # Random Forest
GREEN    = "#16A34A"   # XGBoost
RED_LINE = "#DC2626"   # Perfect Prediction / đường tham chiếu

MODEL_COLORS = {"Linear Regression": ACCENT,"Random Forest": ORANGE,"XGBoost": GREEN}
PERFECT_LINE_COLOR = RED_LINE

RAW_DATA_PATH = "/content/data_public.csv"
df = pd.read_csv(RAW_DATA_PATH)
df_raw = df.copy()

## 1.1. Giới thiệu đề tài

Thị trường bất động sản TP. Hồ Chí Minh có quy mô lớn và giá đăng bán chịu tác động đồng thời bởi vị trí, diện tích, loại hình, cấu hình phòng và các đặc điểm của bất động sản. Vì vậy, xây dựng một công cụ ước lượng giá tham khảo từ dữ liệu có thể hỗ trợ người mua, nhà đầu tư và đơn vị môi giới trong bước sàng lọc và ra quyết định ban đầu.

Đề tài tập trung xây dựng mô hình học máy để **dự đoán tổng giá đăng bán** của hai loại hình thực tế có trong tập dữ liệu là **Căn hộ chung cư** và **Nhà riêng** tại TP. Hồ Chí Minh. Biến mục tiêu gốc là `Price` (triệu VND) và được biến đổi thành `log_price = log1p(Price)` khi huấn luyện nhằm giảm ảnh hưởng của phân phối lệch phải. Sau dự đoán, kết quả được chuyển ngược về đơn vị triệu VND để diễn giải.

Đơn giá theo mét vuông (`Price/Area`) **không phải biến mục tiêu của mô hình**. Biến này chỉ được sử dụng trong EDA, kiểm tra tính hợp lý của dữ liệu và xây dựng quy tắc lọc bất thường.

## 1.2. Câu hỏi nghiên cứu

* Những đặc trưng nào đóng góp nhiều nhất vào dự đoán tổng giá đăng bán của bất động sản?
* Trong ba mô hình Linear Regression, Random Forest và XGBoost, mô hình nào dự đoán `log_price` tốt nhất trên dữ liệu chưa được sử dụng để huấn luyện?

## 1.3. Mục tiêu nghiên cứu

* Xây dựng quy trình làm sạch và tạo đặc trưng cho dữ liệu nhà ở dân dụng tại TP. Hồ Chí Minh.
* Xây dựng mô hình dự đoán **tổng giá đăng bán** thông qua biến mục tiêu `log_price`.
* So sánh Linear Regression, Random Forest và XGBoost bằng Cross-Validation và tập Test độc lập.
* Đánh giá mô hình bằng R², MAE và RMSE trên `log_price`, đồng thời báo cáo MAE và MAPE sau khi chuyển về giá thật.
* Phân tích mức đóng góp của các đặc trưng bằng hệ số mô hình, Feature Importance và SHAP để hỗ trợ diễn giải kết quả.

**Phạm vi ứng dụng:** Kết quả là mức giá đăng tham khảo trong phạm vi dữ liệu nghiên cứu, không thay thế hoạt động thẩm định giá chuyên nghiệp.

**Nguồn dữ liệu:** Kaggle — *Ho Chi Minh City Real Estate Data 2025*  

# A. EDA & TRỰC QUAN HÓA
## A1. Phân tích thống kê mô tả

Mục tiêu: xác định phạm vi dữ liệu nhà ở dân dụng, kiểm tra chất lượng dữ liệu và mô tả các biến số bằng mean, median, std, min, max, skewness và kurtosis.


## Cô lập phạm vi nghiên cứu — Nhà ở dân dụng (`df_residential`)

**Lý do:** Đề tài tập trung dự đoán giá **nhà ở dân dụng** (Nhà riêng, Nhà mặt tiền, Căn hộ chung cư, Biệt thự liền kề, Nhà biệt thự), không bao gồm Đất, Khách sạn, Nhà trọ, Văn phòng... — các loại hình này có cơ chế định giá hoàn toàn khác (theo m² đất thô, theo phòng kinh doanh...) và sẽ gây nhiễu thống kê nếu gộp chung.


In [ ]:
# Cô lập phân khúc Nhà ở dân dụng
RESIDENTIAL_TYPES = ["Nhà riêng", "Nhà mặt tiền", "Căn hộ chung cư","Biệt thự liền kề", "Nhà biệt thự",]
df_residential = df_raw[df_raw["Property Type"].isin(RESIDENTIAL_TYPES)].copy()

print(f"df_raw          : {df_raw.shape[0]:,} dòng (toàn bộ loại hình BĐS)")
print(f"df_residential  : {df_residential.shape[0]:,} dòng "f"({df_residential.shape[0] / df_raw.shape[0] * 100:.1f}% của df_raw)")
print()
print("Phân bố Property Type trong df_residential:")
print(df_residential["Property Type"].value_counts())
print()
print("Phân bố Property Type bị loại (không thuộc Nhà ở dân dụng):")
print(df_raw.loc[~df_raw["Property Type"].isin(RESIDENTIAL_TYPES), "Property Type"].value_counts())

In [ ]:
df = df_residential.copy()
print(f"Kích thước dữ liệu nghiên cứu: {df.shape[0]:,} dòng × {df.shape[1]} cột")
df.info()

In [ ]:
# Thống kê mô tả trên đúng phạm vi Nhà ở dân dụng
numeric_cols_desc = df.select_dtypes(include=np.number).columns.tolist()
descriptive_stats = df[numeric_cols_desc].describe().T
descriptive_stats["median"] = df[numeric_cols_desc].median()
descriptive_stats["skewness"] = df[numeric_cols_desc].skew()
descriptive_stats["kurtosis"] = df[numeric_cols_desc].kurtosis()

descriptive_stats = descriptive_stats[[
    "count", "mean", "median", "std", "min",
    "25%", "50%", "75%", "max", "skewness", "kurtosis"]]

display(descriptive_stats.round(3))

### Nhận xét A1

- Mean của Price/Area lớn hơn rất nhiều so với median => phân phối bị kéo lệch bởi một số quan sát có giá trị cực lớn.
- Skewness và kurtosis cực cao cho thấy dữ liệu lệch phải nặng.
- Lỗi gõ thừa số 0 hoặc nhập sai đơn vị ở Price và Area, lỗi nhập số âm ở biến floors.

→ **Quyết định:** áp dụng log-transform (kéo phân phối lệch phải về gần phân phối chuẩn); kiểm tra bất thường trước khi huấn luyện.


## A2. Trực quan hóa dữ liệu



In [ ]:
# Bảng missing value chi tiết từng cột (tính trên df_residential)
missing_count = df.isnull().sum()
missing_pct = (missing_count / len(df) * 100).round(2)
missing_table = pd.DataFrame({"missing_count": missing_count,"missing_pct": missing_pct}).sort_values("missing_pct", ascending=False)

n_cols_missing = (missing_count > 0).sum()
print(f"Số cột có missing value: {n_cols_missing}/{df.shape[1]}")
missing_table[missing_table["missing_count"] > 0]

In [ ]:
# Biểu đồ 1: Bar chart % missing theo cột, sắp xếp giảm dần
miss_plot = missing_table[missing_table["missing_count"] > 0]

plt.figure(figsize=(10, 7))
sns.barplot(x=miss_plot["missing_pct"], y=miss_plot.index, hue=miss_plot.index,palette="Reds_r", legend=False)
plt.axvline(30, color="black", linestyle="--", linewidth=1, label="Ngưỡng 30%")
plt.title("EDA-01 · Tỷ lệ Missing Value theo Cột — df_residential (giảm dần)", fontsize=13, fontweight="bold")
plt.xlabel("% Missing")
plt.ylabel("")
plt.legend()
for i, v in enumerate(miss_plot["missing_pct"]):
    plt.text(v + 0.5, i, f"{v}%", va="center", fontsize=8)
plt.tight_layout()
plt.show()

### Nhận xét Bar chart missing

- **14/28 cột** chứa giá trị thiếu, trong đó:
  - **Road Type:** 85,30%
  - **Alley Width:** 84,68%
  - **Direction:** 77,49%
  - **GPS:** 72,88%

→ **Thiếu tập trung ở nhóm mô tả chi tiết**, nên nhóm không xóa dòng mà điền các thông số thống kê theo **Train** và tạo cờ `*_is_missing` (vì nó phản ánh hành vi của người đăng tin và đặc điểm của BĐS đó).


In [ ]:
# Biểu đồ 2: Histogram + KDE — full range (trái) vs zoom vào vùng phổ biến (phải)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["Price"], bins=60, kde=False, ax=axes[0], color="indianred")
axes[0].set_title("Price — Toàn bộ phạm vi (df_residential, thô)", fontweight="bold")
axes[0].set_xlabel("Price (triệu VND)")

zoom = df.loc[df["Price"] <= 50_000, "Price"]  # vùng <= 50 tỷ, ngưỡng cứng dự kiến ở S4
sns.histplot(zoom, bins=60, kde=True, ax=axes[1], color="steelblue")
axes[1].set_title(f"Price — Zoom <= 50.000 triệu ({len(zoom):,}/{len(df):,} dòng, "f"{len(zoom)/len(df)*100:.1f}%)", fontweight="bold")
axes[1].set_xlabel("Price (triệu VND)")

plt.suptitle("EDA-02 · Phân phối Price thô (df_residential) — Histogram + KDE", fontweight="bold")
plt.tight_layout()
plt.show()

### Nhận xét Biểu đồ 2 (Histogram Price)

- **Khung trái:** Toàn bộ dữ liệu thô bị dồn nén thành một cột duy nhất sát trục tung. Trục hoành bị kéo giãn đến mức **1.6×10⁷ triệu đồng** (tương đương **16.000 tỷ VNĐ**) => che phân phối thực tế.
- **Khung phải (zoom Price ≤ 50.000tr):** Phân phối phổ biến hiện rõ, vẫn lệch phải, vì vậy nhóm chọn dự đoán **`log_price`**; ngưỡng **Price 100 đến dưới 50.000 triệu** (giữ lại **86% vùng dữ liệu thực tế**).


In [ ]:
# Biểu đồ 3: Boxplot tổng giá — định lượng outlier theo IQR
Q1, Q3 = df["Price"].quantile([0.25, 0.75])
IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR
n_outliers = (df["Price"] > upper_fence).sum()

plt.figure(figsize=(10, 3))
sns.boxplot(x=df["Price"], color="lightcoral")
plt.title("EDA-03 · Boxplot Price (df_residential, thô) — toàn bộ phạm vi", fontweight="bold")
plt.xlabel("Price (triệu VND)")
plt.tight_layout()
plt.show()

print(f"Q1={Q1:,.0f} | Q3={Q3:,.0f} | IQR={IQR:,.0f} | Upper fence (Q3+1.5*IQR)={upper_fence:,.0f}")
print(f"Số điểm vượt upper fence: {n_outliers:,} ({n_outliers/len(df)*100:.2f}%)")

### Nhận xét Biểu đồ 3 (Boxplot Price)

- Q1 = 3.790, Q3 = 20.000, IQR = 16.210 → ngưỡng trên (Q3 + 1.5×IQR) = 44.315 triệu.
- Có tới **2.925 điểm (15,24%)** vượt ngưỡng trên — tỷ lệ outlier khá cao theo chuẩn IQR.
- Vì tỷ lệ này lớn, nếu loại bỏ theo IQR sẽ mất một lượng dữ liệu đáng kể; đây là lý do notebook chọn áp dụng ngưỡng nghiệp vụ thay vì cắt cứng theo IQR trước khi chia Train/Test.
- Phần lớn **Price** tập trung ở vùng thấp; nhiều điểm kéo dài về bên phải → đuôi phải dài, xuất hiện nhiều giá trị cực đoan.
- **BĐS giá cao có thể là dữ liệu hợp lệ.**
→ **IQR** dùng để phát hiện bất thường; **ngưỡng nghiệp vụ** mới được dùng để quyết định làm sạch.

In [ ]:
# Biểu đồ 4: Histogram diện tích — full range (trái) vs zoom (phải)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["Area"], bins=60, ax=axes[0], color="indianred")
axes[0].set_title("Area — Toàn bộ phạm vi (df_residential, thô)", fontweight="bold")
axes[0].set_xlabel("Area (m²)")

zoom_area = df.loc[df["Area"] <= 1_000, "Area"]
sns.histplot(zoom_area, bins=60, ax=axes[1], color="steelblue")
axes[1].set_title(f"Area — Zoom <= 1.000 m² ({len(zoom_area):,}/{len(df):,} dòng, "f"{len(zoom_area)/len(df)*100:.1f}%)", fontweight="bold")
axes[1].set_xlabel("Area (m²)")

plt.suptitle("EDA-04 · Phân phối Area thô (df_residential) — Histogram", fontweight="bold")
plt.tight_layout()
plt.show()

### Nhận xét Biểu đồ 4 (Histogram Area)

- Diện tích có xu hướng lệch phải: phần lớn tin đăng nằm ở nhóm diện tích phổ biến, trong khi một số ít quan sát có diện tích rất lớn.
- Khung phóng đại giúp quan sát rõ vùng dữ liệu chính; phạm vi đầy đủ giúp nhận diện phần đuôi và các trường hợp bất thường.
- Ngưỡng diện tích được xử lý theo quy tắc nghiệp vụ trước khi chia dữ liệu.


In [ ]:
print("Số giá trị duy nhất của Bedrooms:", df['Bedrooms'].nunique())
print(f"Bedrooms = 0      : {(df['Bedrooms'] == 0).sum()} dòng")
print(f"Bedrooms > 10     : {(df['Bedrooms'] > 10).sum()} dòng "f"({(df['Bedrooms'] > 10).mean()*100:.2f}% trên phần có giá trị)")
print(f"Giá trị Bedrooms lớn nhất: {df['Bedrooms'].max():.0f}")
print("Top 10 giá trị Bedrooms phổ biến nhất:")
print(df['Bedrooms'].value_counts().head(10))

In [ ]:
# Biểu đồ 5: Countplot Bedrooms — giới hạn hiển thị 0-10 cho dễ đọc, ghi chú phần ngoài range
plt.figure(figsize=(10, 5))
bedrooms_capped = df["Bedrooms"].clip(upper=10)
order = sorted(bedrooms_capped.dropna().unique())
ax = sns.countplot(x=bedrooms_capped, order=order, color="steelblue")
n_over10 = (df["Bedrooms"] > 10).sum()
labels = [str(int(v)) if v < 10 else "10+" for v in order]
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels)
plt.title(f"EDA-05 · Countplot Bedrooms (df_residential, thô) — giá trị >10 gộp vào '10+' ({n_over10} dòng)", fontweight="bold")
plt.xlabel("Số phòng ngủ")
plt.ylabel("Số tin đăng")
plt.tight_layout()
plt.show()

### Nhận xét Biểu đồ 5 (Countplot Bedrooms)

- Phân bố tập trung mạnh ở **2 phòng ngủ (7.259 tin, ~46% dữ liệu có giá trị)**, giảm dần ở 3 phòng (3.702), 4 phòng (1.722) và 1 phòng (1.460).
- Giá trị Bedrooms lớn nhất ghi nhận là **6.902** — chắc chắn là lỗi nhập liệu (không có bất động sản dân dụng nào có 6.902 phòng ngủ). Notebook xử lý hợp lý bằng cách clip hiển thị ở mức "10+" (257 dòng, 1,34%) thay vì loại bỏ dữ liệu gốc.
- Kết luận: nhóm nhà ở phổ biến nhất là 2–3 phòng ngủ, phù hợp với phân khúc căn hộ/nhà phố vừa và nhỏ.


In [ ]:
print("Các giá trị Floors khác thường (ngoài khoảng 0-20):")
print(sorted(df.loc[(df['Floors'] < 0) | (df['Floors'] > 20), 'Floors'].unique()))
print(f"Floors < 0  : {(df['Floors'] < 0).sum()} dòng")
print(f"Floors > 20 : {(df['Floors'] > 20).sum()} dòng")

In [ ]:
# Biểu đồ 6: Countplot Floors — giới hạn hiển thị 0-15 cho dễ đọc
plt.figure(figsize=(10, 5))
floors_valid_range = df.loc[(df["Floors"] >= 0) & (df["Floors"] <= 15), "Floors"]
n_outside = ((df["Floors"] < 0) | (df["Floors"] > 15)).sum()
order = sorted(floors_valid_range.dropna().unique())
ax = sns.countplot(x=floors_valid_range, order=order, color="darkorange")
plt.title(f"EDA-06 · Countplot Floors (df_residential, thô) — phạm vi 0-15 tầng ({n_outside} dòng ngoài phạm vi không hiển thị)",fontweight="bold")
plt.xlabel("Số tầng")
plt.ylabel("Số tin đăng")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Nhận xét Biểu đồ 6 (Countplot Floors)

- Số tầng tập trung mạnh ở **2, 3 và 4 tầng** — phù hợp với đặc trưng nhà phố/nhà ống của TP.HCM.
- Có 3 dòng Floors âm (-1, -2) và 6 dòng > 20 tầng, trong đó có giá trị **999999.99** — rõ ràng là sentinel lỗi crawl chứ không phải số tầng thật, cần được coi là missing/error thay vì giá trị hợp lệ.
- Các giá trị khác thường này cần được đối chiếu thêm với loại hình và mô tả tin đăng trước khi quyết định impute hay loại bỏ.

In [ ]:
# EDA-06.2: So sánh Price/Area giữa các loại hình bằng groupby
compare_table = df.groupby("Property Type").agg(
    n_listings=("Price", "count"),
    price_mean=("Price", "mean"),
    price_median=("Price", "median"),
    price_std=("Price", "std"),
    area_mean=("Area", "mean"),
    area_median=("Area", "median"),
    area_std=("Area", "std"),
).round(1).sort_values("n_listings", ascending=False)

# Đơn giá / m2 để so sánh công bằng hơn giữa các loại hình có diện tích khác nhau
compare_table["unit_price_median"] = (
    df.assign(unit_price=df["Price"] / df["Area"])
    .groupby("Property Type")["unit_price"].median()
    .round(2))

print("So sánh Price / Area / Đơn giá theo Property Type:")
print(compare_table)

In [ ]:
# EDA-06.3: Boxplot so sánh Price và Đơn giá/m2 giữa các loại hình
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_plot = df.copy()
df_plot["log_price"] = np.log1p(df_plot["Price"])
df_plot["unit_price"] = df_plot["Price"] / df_plot["Area"]
df_plot["log_unit_price"] = np.log1p(df_plot["unit_price"])

order = df_plot["Property Type"].value_counts().index

sns.boxplot(data=df_plot, x="Property Type", y="log_price", order=order, ax=axes[0])
axes[0].set_title("log(Price) theo Property Type", fontweight="bold")
axes[0].tick_params(axis="x", rotation=30)
sns.boxplot(data=df_plot, x="Property Type", y="log_unit_price", order=order, ax=axes[1])
axes[1].set_title("log(Đơn giá / m²) theo Property Type", fontweight="bold")
axes[1].tick_params(axis="x", rotation=30)

plt.suptitle("EDA-06 — So sánh Giá giữa các loại hình Nhà ở dân dụng (df_residential)", fontweight="bold")
plt.tight_layout()
plt.show()

### Nhận xét Biểu đồ EDA-06.3 (Boxplot theo Property Type)

- Nhà riêng có xu hướng có giá cao và ổn định hơn căn hộ.
- Giá giữa các căn hộ **không đồng đều**, mức biến động lớn hơn nhà riêng.
- Cả 2 loại hình đều có **nhiều giá trị outlier**.
- Kết quả này chứng minh Property Type là biến phân loại có ảnh hưởng đáng kể đến giá, cần đưa vào mô hình dưới dạng one-hot hoặc biến phân nhóm.

In [ ]:
# Biểu đồ 8: Ma trận tương quan các biến số chính
corr_cols = [c for c in ["Price", "Area", "Bedrooms", "Bathrooms", "Floors", "Latitude", "Longitude"] if c in df.columns]
plt.figure(figsize=(9, 6))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Tương quan giữa các biến số chính")
plt.xlabel("Biến")
plt.ylabel("Biến")
plt.tight_layout()
plt.show()

### Nhận xét Biểu đồ 8 (Heatmap tương quan)

- Bedrooms–Bathrooms có tương quan dương rõ nhất trong nhóm biến số (**r = 0,68**), vì bất động sản nhiều phòng ngủ thường đi kèm nhiều phòng tắm.
- Bedrooms–Floors tương quan dương ở mức trung bình (**r = 0,31**).
- Tương quan tuyến tính của Price với Area, Latitude, Longitude đều rất yếu (gần 0,00) — cho thấy quan hệ giá–vị trí/diện tích ở TP.HCM mang tính phi tuyến/theo cụm, không thể nắm bắt bằng hệ số tương quan Pearson đơn giản.
- Đây là lý do mô hình sử dụng thêm các đặc trưng dẫn xuất như `location_rank`, `geo_region_id` và `distance_to_center` thay vì dùng trực tiếp toạ độ thô.

### Nhận xét A2

- Missing value tập trung ở nhóm đặc điểm chi tiết của bất động sản, vì vậy cần điền khuyết thay vì xóa hàng loạt.
- Price và Area có xu hướng lệch phải; biến đổi log giúp mô hình ổn định hơn.
- Phân phối số phòng và số tầng cho thấy tồn tại một nhóm nhà ở phổ biến cùng một số trường hợp hiếm.
- Đơn giá và tổng giá khác biệt theo loại hình, cho thấy `Property Type` cần được giữ trong mô hình.


## A3. Insight kinh doanh từ EDA

1. **Phân khúc phổ biến:** Doanh nghiệp nên ưu tiên nguồn tin và chiến dịch quảng cáo cho nhóm nhà ở xuất hiện thường xuyên nhất trong biểu đồ số phòng, số tầng và diện tích.
2. **Giá khác biệt theo vị trí và loại hình:** Cùng quy mô nhưng bất động sản ở khu vực hoặc loại hình khác nhau có thể có mặt bằng giá khác nhau; công cụ định giá cần kết hợp cả vị trí và `Property Type`.
3. **Dữ liệu thiếu phản ánh cấu trúc tin đăng:** Việc thiếu hướng, độ rộng hẻm hoặc số tầng có thể mang thông tin về loại hình tài sản, vì vậy các cờ missing được giữ làm đặc trưng.
4. **Tin cực đoan cần được kiểm duyệt:** Các tin có giá, diện tích hoặc đơn giá bất thường nên được kiểm tra trước khi dùng để huấn luyện hay hiển thị cho người dùng.


# B. XÂY DỰNG MÔ HÌNH MACHINE LEARNING
## B1. Tiền xử lý dữ liệu

Quy trình gồm loại cột không cần thiết, chuẩn hóa kiểu dữ liệu, tách Train/Test trước khi tính thống kê, xử lý missing, outlier và duplicate, tạo đặc trưng vị trí, encoding và scaling. Mọi tham số học từ dữ liệu chỉ được fit trên tập Train để tránh rò rỉ dữ liệu.


# B1. Quy trình tiền xử lý dữ liệu — theo đúng luồng code

## 1. Tạo binary flags từ văn bản (Title/Description)

Dùng regex tìm từ khóa trong `Title` và `Description` để tạo 5 flag nhị phân:

- `has_furniture` — có nội thất
- `car_alley` — hẻm ô tô vào được
- `near_market` — gần chợ
- `near_school` — gần trường học
- `is_urgent_sale` — bán gấp / chính chủ

## 2. Chuẩn hóa ngày cập nhật → `listing_age`

Parse `Last Updated Date` (format `%d/%m/%Y %H:%M`), sau đó tính:
```
listing_age = (CRAWL_DATE - Last Updated Date).days
```
với `CRAWL_DATE = 2025-09-30`

## 3. Trích xuất `location_area` từ `Location`

- Chuẩn hóa text (Unicode, xóa khoảng trắng thừa).
- Mapping Phường/Xã → Quận/Huyện bằng dictionary `WARD_TO_AREA`.
- Gộp Quận 2 + Quận 9 + Thủ Đức cũ → **TP. Thủ Đức**
Mục đích chính của bước tạo **`location_area`** là **chuyển đổi địa chỉ thô dạng chữ tự do (vốn cực kỳ lộn xộn, máy không thể đọc được) thành một biến phân loại hành chính chuẩn hóa (cấp Quận/Huyện)**.

Bước này phục vụ **2 nhiệm vụ** cho các bước sau:

1. Dùng để điền khuyết cho **72,88% tọa độ GPS bị thiếu** (bằng cách lấy tọa độ trung vị của quận đó).
2. Dùng để tính toán độ đắt đỏ của từng khu vực (`location_rank`), giúp mô hình hiểu được giá trị khác biệt giữa các quận (ví dụ: Quận 1 đắt hơn Quận Bình Tân thế nào)

## 4. Lọc ngưỡng cố định & deduplication (TRƯỚC Train/Test Split)

- Địa chỉ "TP.HCM_Mới" chưa xác định rõ → gom vào nhóm `Unknown` thay vì xóa.
- Loại rõ các địa chỉ ngoài TP.HCM (VD: chứa "Hà Nội").
- Chuyển giá trị sentinel/phi thực tế thành `NaN` bằng ngưỡng cố định:
  - `Length > 100`, `Alley Width > 30`
  - Tọa độ ngoài khoảng `Latitude [10.3, 11.2]`, `Longitude [106.3, 107.0]`
- Lọc theo ngưỡng nghiệp vụ cố định (áp dụng trước split vì đây là ngưỡng biết trước, không học từ phân phối dữ liệu):
  - `Price` trong [100, 50.000) triệu VND
  - `Area` trong [5, 500) m²
  - Đơn giá (`Price/Area`) ≥ 10 triệu/m²
  - `Bedrooms` trong [1,10], `Floors` trong [0,100]
- Dedup: sắp xếp theo `Last Updated Date` giảm dần, ưu tiên giữ theo `Listing ID` (bản ghi mới nhất), phần không có ID xử lý riêng.

## 5. Train/Test Split

- `train_test_split(test_size=0.2, random_state=42, stratify=Property Type)` — stratify để giữ tỷ lệ phân khúc bất động sản đồng đều giữa hai tập.
- Reset index cho `df_train`, `df_test`.

## 6. Xử lý Missing Value — chỉ fit thống kê trên Train

- Tạo **missing flag** cho từng cột trước khi impute: `STRUCT_COLS` (Bedrooms, Floors, Bathrooms), `SPATIAL_COLS` (Width, Length, Alley Width), `CAT_COLS_RAW` (Direction, Road Type, Position) → điền `"Unknown"` cho cột phân loại.
- Impute `Bedrooms/Floors/Bathrooms`: median theo nhóm `Property Type` (học trên Train), fallback về median toàn cục nếu nhóm vẫn thiếu.
- `Alley Width = 0` cho loại hình không có hẻm (`Nhà mặt tiền`, `Căn hộ chung cư`).
- Impute `Width/Length/Alley Width` còn lại: median theo `Property Type`, học trên Train, áp dụng cho cả Train và Test.

## 7. Feature Engineering

- `log_price = log1p(Price)`, `log_area = log1p(Area)` — giảm lệch phải (right-skew).
- `road_width_bin`: phân loại `Alley Width` → `no_alley / narrow (<4) / medium (4–8) / wide (>8) / unknown`.
- `district_tier`: `central / suburban / outer` — gán theo tập hợp Quận/Huyện **cố định**, không dùng `Price` (tránh leakage).
- `distance_to_center`: khoảng cách Haversine từ mỗi bất động sản đến trung tâm TP.HCM (10.7769, 106.7009).

## 8. `geo_region_id` — Phân cụm không gian bằng KMeans

- Chuyển tọa độ sang radian, chỉ dùng tọa độ của **Train** để dò K.
- Elbow Method (WCSS) + Silhouette Score cho K từ 2 đến 20.
- Silhouette cao nhất tại **K=20** — nhưng đây cũng là biên trên của dải khảo sát nên chưa chắc là điểm dừng tối ưu tuyệt đối; Elbow thực chất "gãy khuỷu" ở K≈4–5.
- Chọn `K_BEST = 20`, fit KMeans trên Train → Silhouette = 0,7602. Các cụm phân chia khá rõ theo khu vực địa lý, ít chồng lấn.
- **Test chỉ được gán cụm bằng `.predict()` của mô hình đã fit trên Train** — không có rò rỉ dữ liệu.
- `geo_region_id` dùng như biến phân loại (One-Hot), không phải biến số có thứ tự.

## 9. `location_rank` — Out-of-Fold Target Encoding

- Vấn đề: nếu encode trực tiếp bằng thống kê của toàn bộ Train rồi dùng lại cho chính Train → leakage.
- Giải pháp: **KFold 5 fold** (`shuffle=True, random_state=42`). Với mỗi fold: tính median `log_price` theo `location_area` trên 4 fold "fit", map rank (1 = giá thấp nhất) vào fold "valid" còn lại — dòng nào cũng được rank **mà không dùng chính giá trị của nó**.
- Fallback: nếu khu vực chưa gặp trong fold fit → gán median của các rank hiện có.
- Test: map bằng thống kê tính trên **toàn bộ Train** (không cần OOF vì Test không tham gia fit).

## 10. Hoàn thiện `train_df` / `test_df`

- Gán `geo_region_id` (nhãn cụm) vào cả hai tập.
- Drop các cột đã "chuyển hóa" thành đặc trưng khác, không đưa trực tiếp vào X: `Price, Area, Latitude, Longitude, lat_is_missing`.
- Assert: cột giữa Train/Test khớp nhau; các đặc trưng chính (`listing_age`, `location_rank`, `district_tier`, `geo_region_id`) đã tồn tại.

---

## Kết luận B1

- Ngưỡng nghiệp vụ và deduplication thực hiện **trước** Train/Test Split.
- Mọi median và centroid chỉ fit trên Train rồi áp dụng cho Test.
- `district_tier` xác định theo địa lý cố định, không dựa vào giá.
- `location_rank` tạo theo Out-of-Fold trên Train để hạn chế target leakage.
- `geo_region_id` tạo từ phân cụm tọa độ trên Train, dùng như biến phân loại.
- Toàn bộ đặc trưng trong `LINREG_FEATURES` và `RF_FEATURES` được giữ lại, sẵn sàng cho bước Xây dựng mô hình (B2).

In [ ]:
# Hàm trợ giúp: tìm pattern trong cả Title lẫn Description
def flag_keywords(df, col1, col2, patterns):
    """Trả về Series 0/1 nếu tìm thấy bất kỳ pattern nào trong col1 hoặc col2."""
    s1 = df[col1].fillna("").str.lower()
    s2 = df[col2].fillna("").str.lower()
    regex = "|".join(patterns)
    return ((s1.str.contains(regex, regex=True)) |(s2.str.contains(regex, regex=True))).astype(int)

# Tạo 5 binary flags
# Flag 1: has_furniture — BĐS có nội thất → cao hơn 5–15% so với nhà trống
df["has_furniture"] = flag_keywords(df, "Title", "Description",[r"nội thất", r"full nội thất", r"đầy đủ nội thất", r"tặng nội thất", r"nội thất cao cấp"])

# Flag 2: car_alley — Hẻm xe hơi → chênh lệch giá 20–40% so với hẻm xe máy
df["car_alley"] = flag_keywords(df, "Title", "Description",[r"hẻm xe hơi", r"ô tô vào", r"đỗ xe", r"oto vào", r"hxh"])

# Flag 3: near_market — Gần chợ → tiện lợi sinh hoạt tác động tích cực đến giá
df["near_market"] = flag_keywords(df, "Title", "Description",[r"gần chợ", r"sát chợ", r"trước chợ"])

# Flag 4: near_school — Gần trường → gia đình sẵn sàng trả premium 10–20%
df["near_school"] = flag_keywords(df, "Title", "Description",[r"gần trường", r"khu giáo dục",r"đại học", r"trường học", r"trường tiểu học", r"trường thcs"])

# Flag 5: is_urgent_sale — "gấp/chính chủ" → thường thấp hơn thị trường 5–10%
df["is_urgent_sale"] = flag_keywords(df, "Title", "Description",[r"cần bán gấp", r"chính chủ", r"giá tốt", r"bán gấp"])

# Kiểm tra tỷ lệ mỗi flag
flag_cols = ["has_furniture", "car_alley", "near_market", "near_school", "is_urgent_sale"]
print("Tỷ lệ tin đăng có từng flag (%):")
for col in flag_cols:
    pct = df[col].mean() * 100
    print(f"  {col:<20}: {df[col].sum():>6,} dòng ({pct:.1f}%)")

In [ ]:
# Chuẩn hóa ngày cập nhật và tạo listing_age
df["Last Updated Date"] = pd.to_datetime(
    df["Last Updated Date"],
    format="%d/%m/%Y %H:%M",
    errors="coerce")

CRAWL_DATE = pd.Timestamp("2025-09-30")
df["listing_age"] = (CRAWL_DATE - df["Last Updated Date"]).dt.days.clip(lower=0)

print("Khoảng Last Updated Date:",df["Last Updated Date"].min(), "→", df["Last Updated Date"].max())
print("Số lỗi parse:", df["Last Updated Date"].isna().sum())

In [ ]:
# 2d. Trích xuất Location Area từ Location
# Mục tiêu:
# - Chuẩn hóa Location thành đơn vị hành chính cấp Quận/Huyện
# - Gộp Quận 2 + Quận 9 + Quận Thủ Đức -> TP. Thủ Đức
# 1. HÀM CHUẨN HÓA TEXT
def normalize_text(text):
    if pd.isna(text):
        return ""
    text = unicodedata.normalize("NFC", str(text))
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# 2. MAPPING PHƯỜNG/XÃ -> QUẬN/HUYỆN
WARD_TO_AREA = {
    # TP THỦ ĐỨC
    "an khánh":"TP. Thủ Đức",
    "an lợi đông":"TP. Thủ Đức",
    "an phú":"TP. Thủ Đức",
    "bình chiểu":"TP. Thủ Đức",
    "bình thọ":"TP. Thủ Đức",
    "bình trưng đông":"TP. Thủ Đức",
    "bình trưng tây":"TP. Thủ Đức",
    "cát lái":"TP. Thủ Đức",
    "hiệp bình":"TP. Thủ Đức",
    "hiệp bình chánh":"TP. Thủ Đức",
    "hiệp bình phước":"TP. Thủ Đức",
    "hiệp phú":"TP. Thủ Đức",
    "linh chiểu":"TP. Thủ Đức",
    "linh đông":"TP. Thủ Đức",
    "linh tây":"TP. Thủ Đức",
    "linh trung":"TP. Thủ Đức",
    "linh xuân":"TP. Thủ Đức",
    "long bình":"TP. Thủ Đức",
    "long phước":"TP. Thủ Đức",
    "long thạnh mỹ":"TP. Thủ Đức",
    "long trường":"TP. Thủ Đức",
    "phú hữu":"TP. Thủ Đức",
    "phước bình":"TP. Thủ Đức",
    "phước long a":"TP. Thủ Đức",
    "phước long b":"TP. Thủ Đức",
    "tam bình":"TP. Thủ Đức",
    "tam phú":"TP. Thủ Đức",
    "thạnh mỹ lợi":"TP. Thủ Đức",
    "thảo điền":"TP. Thủ Đức",
    "thủ thiêm":"TP. Thủ Đức",
    "trường thạnh":"TP. Thủ Đức",
    "trường thọ":"TP. Thủ Đức",
    "tăng nhơn phú a":"TP. Thủ Đức",
    "tăng nhơn phú b":"TP. Thủ Đức",

    # QUẬN 1
    "bến nghé":"Quận 1",
    "bến thành":"Quận 1",
    "cầu kho":"Quận 1",
    "cầu ông lãnh":"Quận 1",
    "cô giang":"Quận 1",
    "đa kao":"Quận 1",
    "nguyễn cư trinh":"Quận 1",
    "nguyễn thái bình":"Quận 1",
    "phạm ngũ lão":"Quận 1",
    "tân định":"Quận 1",

    # QUẬN 3
    "võ thị sáu":"Quận 3",

    # QUẬN 7
    "bình thuận":"Quận 7",
    "phú mỹ":"Quận 7",
    "phú thuận":"Quận 7",
    "tân hưng":"Quận 7",
    "tân kiểng":"Quận 7",
    "tân phong":"Quận 7",
    "tân quy":"Quận 7",
    "tân thuận đông":"Quận 7",
    "tân thuận tây":"Quận 7",

    # QUẬN 12
    "an phú đông":"Quận 12",
    "đông hưng thuận":"Quận 12",
    "hiệp thành":"Quận 12",
    "thạnh lộc":"Quận 12",
    "thạnh xuân":"Quận 12",
    "tân chánh hiệp":"Quận 12",
    "tân hưng thuận":"Quận 12",
    "tân thới hiệp":"Quận 12",
    "tân thới nhất":"Quận 12",
    "trung mỹ tây":"Quận 12",
    "thới an":"Quận 12",

    # QUẬN BÌNH TÂN
    "an lạc":"Quận Bình Tân",
    "an lạc a":"Quận Bình Tân",
    "bình hưng hòa":"Quận Bình Tân",
    "bình hưng hòa a":"Quận Bình Tân",
    "bình hưng hòa b":"Quận Bình Tân",
    "bình trị đông":"Quận Bình Tân",
    "bình trị đông a":"Quận Bình Tân",
    "bình trị đông b":"Quận Bình Tân",
    "tân tạo":"Quận Bình Tân",
    "tân tạo a":"Quận Bình Tân",

    # QUẬN TÂN PHÚ
    "hiệp tân":"Quận Tân Phú",
    "hòa thạnh":"Quận Tân Phú",
    "phú thạnh":"Quận Tân Phú",
    "phú thọ hòa":"Quận Tân Phú",
    "phú trung":"Quận Tân Phú",
    "sơn kỳ":"Quận Tân Phú",
    "tân quý":"Quận Tân Phú",
    "tân sơn nhì":"Quận Tân Phú",
    "tân thành":"Quận Tân Phú",
    "tân thới hòa":"Quận Tân Phú",
    "tây thạnh":"Quận Tân Phú",

    # HUYỆN BÌNH CHÁNH
    "an phú tây":"Huyện Bình Chánh",
    "bình chánh":"Huyện Bình Chánh",
    "bình hưng":"Huyện Bình Chánh",
    "bình lợi":"Huyện Bình Chánh",
    "đa phước":"Huyện Bình Chánh",
    "hưng long":"Huyện Bình Chánh",
    "lê minh xuân":"Huyện Bình Chánh",
    "phong phú":"Huyện Bình Chánh",
    "tân kiên":"Huyện Bình Chánh",
    "tân nhựt":"Huyện Bình Chánh",
    "tân quý tây":"Huyện Bình Chánh",
    "vĩnh lộc a":"Huyện Bình Chánh",
    "vĩnh lộc b":"Huyện Bình Chánh",

    # HUYỆN CỦ CHI
    "an nhơn tây":"Huyện Củ Chi",
    "bình mỹ":"Huyện Củ Chi",
    "củ chi":"Huyện Củ Chi",
    "hòa phú":"Huyện Củ Chi",
    "nhuận đức":"Huyện Củ Chi",
    "phú hòa đông":"Huyện Củ Chi",
    "phước hiệp":"Huyện Củ Chi",
    "tân an hội":"Huyện Củ Chi",
    "tân phú trung":"Huyện Củ Chi",
    "tân thạnh đông":"Huyện Củ Chi",
    "tân thạnh tây":"Huyện Củ Chi",
    "thái mỹ":"Huyện Củ Chi",
    "trung lập thượng":"Huyện Củ Chi",

    # HUYỆN HÓC MÔN
    "bà điểm":"Huyện Hóc Môn",
    "đông thạnh":"Huyện Hóc Môn",
    "nhị bình":"Huyện Hóc Môn",
    "tân hiệp":"Huyện Hóc Môn",
    "tân xuân":"Huyện Hóc Môn",
    "trung chánh":"Huyện Hóc Môn",
    "xuân thới thượng":"Huyện Hóc Môn",
    "hóc môn":"Huyện Hóc Môn",

    # HUYỆN NHÀ BÈ
    "hiệp phước":"Huyện Nhà Bè",
    "long thới":"Huyện Nhà Bè",
    "nhà bè":"Huyện Nhà Bè",
    "nhơn đức":"Huyện Nhà Bè",
    "phú xuân":"Huyện Nhà Bè",
    "phước kiển":"Huyện Nhà Bè",

    # HUYỆN CẦN GIỜ
    "an thới đông":"Huyện Cần Giờ",
    "bình khánh":"Huyện Cần Giờ",
    "cần thạnh":"Huyện Cần Giờ",
    "long hòa":"Huyện Cần Giờ",
    "lý nhơn":"Huyện Cần Giờ",
    "thạnh an":"Huyện Cần Giờ",}

# 3. HÀM TRÍCH XUẤT
def extract_location_area(location):
    if pd.isna(location):
        return "Unknown"
    s = normalize_text(location)
    parts = [p.strip() for p in s.split(",")]

    # Quận
    for p in parts:
        m = re.fullmatch(r"(?:Quận|Q\.?)\s*(\d+)", p, re.I)
        if m:
            q = int(m.group(1))
            if q in [2, 9]:
                return "TP. Thủ Đức"
            return f"Quận {q}"
        m = re.fullmatch(r"Quận\s+(.+)", p, re.I)
        if m:
            name = m.group(1).strip()
            if "thủ đức" in name.lower():
                return "TP. Thủ Đức"
            return f"Quận {name.title()}"

    # Huyện
    for p in parts:
        m = re.fullmatch(r"Huyện\s+(.+)", p, re.I)
        if m:
            return f"Huyện {m.group(1).strip().title()}"

    # Thành phố Thủ Đức
    if re.search(r"Thủ Đức", s, re.I):
        return "TP. Thủ Đức"

    # Mapping Phường/Xã
    for p in parts:
        clean = re.sub(r"^(Phường|Xã|Thị trấn|P\.|X\.)\s+","", p,flags=re.I).strip()
        key = clean.lower()
        if key in WARD_TO_AREA:
            return WARD_TO_AREA[key]

    # TP.HCM mới
    if "TP. Hồ Chí Minh" in s:
        return "TP.HCM_Moi"
    return "Unknown"

# 4. THỰC THI
df["location_area"] = df["Location"].apply(extract_location_area)

# Xóa Location gốc

# 5. BÁO CÁO
print("KẾT QUẢ LOCATION AREA")
print(df["location_area"].value_counts().head(25))

print(f"\nRows              : {len(df):,}")
print(f"Unique Categories : {df['location_area'].nunique()}")
print(f"Unknown Count     : {(df['location_area']=='Unknown').sum()}")

In [ ]:
# Lọc ngưỡng cố định và deduplication TRƯỚC Train/Test Split
# Giữ các địa chỉ TP.HCM mới nhưng gom vào nhóm Unknown thay vì xóa
df["location_area"] = df["location_area"].replace("TP.HCM_Moi", "Unknown")

# Loại rõ ràng các địa chỉ ngoài TP.HCM
outside_mask = df["Location"].fillna("").str.contains(r"Hà Nội", case=False, regex=True)
print(f"Địa chỉ ngoài phạm vi bị loại: {outside_mask.sum():,} dòng")
df = df.loc[~outside_mask].copy()

# Chuyển sentinel/giá trị phi thực tế thành missing bằng ngưỡng nghiệp vụ cố định
LENGTH_MAX = 100
ALLEY_WIDTH_MAX = 30
LAT_MIN, LAT_MAX = 10.3, 11.2
LON_MIN, LON_MAX = 106.3, 107.0

df.loc[df["Length"] > LENGTH_MAX, "Length"] = np.nan
df.loc[df["Alley Width"] > ALLEY_WIDTH_MAX, "Alley Width"] = np.nan

invalid_gps = (
    (df["Latitude"] < LAT_MIN) | (df["Latitude"] > LAT_MAX) |
    (df["Longitude"] < LON_MIN) | (df["Longitude"] > LON_MAX))
df.loc[invalid_gps, ["Latitude", "Longitude"]] = np.nan

# Các ngưỡng phạm vi cố định phải áp dụng trước split
PRICE_MIN, PRICE_MAX = 100, 50000       # triệu VND
AREA_MIN, AREA_MAX = 5, 500             # m²
UNIT_PRICE_MIN = 10                     # triệu VND/m²

unit_price = df["Price"] / df["Area"].replace(0, np.nan)

valid_mask = (
    df["Price"].ge(PRICE_MIN)
    & df["Price"].lt(PRICE_MAX)
    & df["Area"].ge(AREA_MIN)
    & df["Area"].lt(AREA_MAX)
    & unit_price.ge(UNIT_PRICE_MIN)
    & (df["Bedrooms"].isna() | df["Bedrooms"].between(1, 10))
    & (df["Floors"].isna() | df["Floors"].between(0, 100)))

before_filter = len(df)
df = df.loc[valid_mask].copy()
print(f"Lọc ngưỡng nghiệp vụ: loại {before_filter - len(df):,} dòng")

# Dedup ưu tiên Listing ID, sau đó dùng khóa thuộc tính dự phòng
df = df.sort_values("Last Updated Date", ascending=False)

if "Listing ID" in df.columns:
    has_id = df["Listing ID"].notna()
    df_with_id = df.loc[has_id].drop_duplicates(subset=["Listing ID"], keep="first")
    df_without_id = df.loc[~has_id]
    df = pd.concat([df_with_id, df_without_id], axis=0)

duplicate_key = ["location_area", "Property Type", "Area", "Price","Bedrooms", "Bathrooms", "Last Updated Date"]
duplicate_key = [c for c in duplicate_key if c in df.columns]

before_dedup = len(df)
df = (
    df.sort_values("Last Updated Date", ascending=False)
      .drop_duplicates(subset=duplicate_key, keep="first")
      .copy())
print(f"Deduplication trước split: loại {before_dedup - len(df):,} dòng")

# Bỏ cột không dùng sau khi đã trích xuất thông tin và dedup
drop_columns = [
    "Title", "Description", "Avatar", "Agent Role",
    "Agent Listing Count", "Province", "VIP Account",
    "Scraped At", "Last Updated", "Property Type Slug",
    "Listing ID", "Location", "Agent Name", "Last Updated Date"]
df = df.drop(columns=drop_columns, errors="ignore")

print(f"Dữ liệu sạch trước split: {df.shape[0]:,} dòng × {df.shape[1]} cột")
print("Phân bố location_area:")
print(df["location_area"].value_counts().head(25).to_string())

In [ ]:
# Train/Test Split sau khi hoàn tất các quy tắc lọc và dedup cố định
stratify_target = None
property_counts = df["Property Type"].value_counts()
if len(property_counts) > 1 and property_counts.min() >= 2:
    stratify_target = df["Property Type"]

df_train, df_test = train_test_split(df,test_size=0.2,random_state=42,stratify=stratify_target)

df_train = df_train.reset_index(drop=True).copy()
df_test = df_test.reset_index(drop=True).copy()

print(f"df_train: {df_train.shape[0]:,} dòng")
print(f"df_test : {df_test.shape[0]:,} dòng")

In [ ]:
# Missing-value handling: chỉ fit thống kê trên Train
STRUCT_COLS = ["Bedrooms", "Floors", "Bathrooms"]
SPATIAL_COLS = ["Width", "Length", "Alley Width"]
CAT_COLS_RAW = ["Direction", "Road Type", "Position"]

# Tạo missing flags trước khi impute
for frame in (df_train, df_test):
    for col in STRUCT_COLS + SPATIAL_COLS:
        flag_name = f"{col.lower().replace(' ', '_')}_is_missing"
        frame[flag_name] = frame[col].isna().astype(int)

    for col in CAT_COLS_RAW:
        flag_name = f"{col.lower().replace(' ', '_')}_is_missing"
        frame[flag_name] = frame[col].isna().astype(int)
        frame[col] = frame[col].fillna("Unknown")

    frame["lat_is_missing"] = (frame["Latitude"].isna() | frame["Longitude"].isna()).astype(int)

# Impute Bedrooms/Floors/Bathrooms theo Property Type
struct_group_medians = {
    col: df_train.groupby("Property Type")[col].median()
    for col in STRUCT_COLS}
struct_global_medians = {
    col: df_train[col].median()
    for col in STRUCT_COLS}

for frame in (df_train, df_test):
    for col in STRUCT_COLS:
        frame[col] = frame[col].fillna(frame["Property Type"].map(struct_group_medians[col]))
        frame[col] = frame[col].fillna(struct_global_medians[col])

# Alley Width bằng 0 cho loại hình không có hẻm
no_alley_types = ["Nhà mặt tiền", "Căn hộ chung cư"]
for frame in (df_train, df_test):
    no_alley_mask = frame["Property Type"].isin(no_alley_types)
    frame.loc[no_alley_mask & frame["Alley Width"].isna(),"Alley Width"] = 0

# Impute Width/Length/Alley Width theo Property Type
spatial_group_medians = {
    col: df_train.groupby("Property Type")[col].median()
    for col in SPATIAL_COLS}
spatial_global_medians = {
    col: df_train[col].median()
    for col in SPATIAL_COLS}

for frame in (df_train, df_test):
    for col in SPATIAL_COLS:
        frame[col] = frame[col].fillna(frame["Property Type"].map(spatial_group_medians[col]))
        frame[col] = frame[col].fillna(spatial_global_medians[col])

# listing_age luôn được giữ; impute bằng median Train nếu ngày cập nhật bị thiếu
listing_age_median = df_train["listing_age"].median()
df_train["listing_age"] = df_train["listing_age"].fillna(listing_age_median)
df_test["listing_age"] = df_test["listing_age"].fillna(listing_age_median)

# GPS: centroid theo location_area, fit trên Train
lat_centroid_map = df_train.groupby("location_area")["Latitude"].median()
lon_centroid_map = df_train.groupby("location_area")["Longitude"].median()

lat_global = df_train["Latitude"].median()
lon_global = df_train["Longitude"].median()
if pd.isna(lat_global):
    lat_global = 10.7769
if pd.isna(lon_global):
    lon_global = 106.7009

for frame in (df_train, df_test):
    frame["Latitude"] = frame["Latitude"].fillna(frame["location_area"].map(lat_centroid_map)).fillna(lat_global)
    frame["Longitude"] = frame["Longitude"].fillna(frame["location_area"].map(lon_centroid_map)).fillna(lon_global)

required_after_impute = (STRUCT_COLS + SPATIAL_COLS + CAT_COLS_RAW+ ["listing_age", "Latitude", "Longitude"])
assert df_train[required_after_impute].isna().sum().sum() == 0
assert df_test[required_after_impute].isna().sum().sum() == 0

print("Đã xử lý missing mà không dùng thống kê từ Test.")

In [ ]:
# Feature Engineering
for frame in (df_train, df_test):
    frame["log_price"] = np.log1p(frame["Price"])
    frame["log_area"] = np.log1p(frame["Area"])

# road_width_bin phải dựa trên Alley Width, không dùng Width của bất động sản
def bin_road_width(width):
    if pd.isna(width):
        return "unknown"
    if width <= 0:
        return "no_alley"
    if width < 4:
        return "narrow"
    if width <= 8:
        return "medium"
    return "wide"

for frame in (df_train, df_test):
    frame["road_width_bin"] = frame["Alley Width"].apply(bin_road_width)

# district_tier theo vị trí địa lý cố định, không dùng Price
CENTRAL_AREAS = {
    "Quận 1", "Quận 3", "Quận 4", "Quận 5", "Quận 10",
    "Quận Phú Nhuận", "Quận Bình Thạnh"}
SUBURBAN_AREAS = {
    "Quận 6", "Quận 7", "Quận 8", "Quận 11",
    "Quận Tân Bình", "Quận Tân Phú", "Quận Gò Vấp",
    "TP. Thủ Đức"}
OUTER_AREAS = {
    "Quận 12", "Quận Bình Tân", "Huyện Bình Chánh",
    "Huyện Hóc Môn", "Huyện Củ Chi", "Huyện Nhà Bè",
    "Huyện Cần Giờ"}

def assign_district_tier(area):
    if area in CENTRAL_AREAS:
        return "central"
    if area in SUBURBAN_AREAS:
        return "suburban"
    if area in OUTER_AREAS:
        return "outer"
    return "unknown"

for frame in (df_train, df_test):
    frame["district_tier"] = (frame["location_area"].apply(assign_district_tier))

# Khoảng cách đến trung tâm TP.HCM
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1_rad = np.radians(lat1)
    lat2_rad = np.radians(lat2)
    dlat = lat2_rad - lat1_rad
    dlon = np.radians(lon2 - lon1)
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1_rad) * np.cos(lat2_rad)
        * np.sin(dlon / 2) ** 2)
    return 2 * R * np.arcsin(np.sqrt(a))

CENTER_LAT, CENTER_LON = 10.7769, 106.7009
for frame in (df_train, df_test):
    frame["distance_to_center"] = haversine(frame["Latitude"],frame["Longitude"],CENTER_LAT,CENTER_LON)

print("Đã tạo log_price, log_area, road_width_bin, district_tier và distance_to_center.")
print("listing_age luôn được giữ trong bộ đặc trưng.")

In [ ]:
# geo_region_id bằng phân cụm không gian: fit trên Train, predict Test
coord_cols = ["Latitude", "Longitude"]
train_coords = np.radians(df_train[coord_cols].to_numpy())
test_coords = np.radians(df_test[coord_cols].to_numpy())

# Elbow + Silhouette cho K từ 2 đến 20
K_RANGE = range(2, 21)
wcss, sil_scores = [], []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_k = km.fit_predict(train_coords)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(train_coords, labels_k))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(K_RANGE), wcss, "bo-", linewidth=2, markersize=5)
axes[0].set_xlabel("Số cluster K", fontsize=11)
axes[0].set_ylabel("WCSS (Inertia)", fontsize=11)
axes[0].set_title("Elbow Method", fontsize=13, fontweight="bold")
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(K_RANGE), sil_scores, "rs-", linewidth=2, markersize=5)
axes[1].set_xlabel("Số cluster K", fontsize=11)
axes[1].set_ylabel("Silhouette Score", fontsize=11)
axes[1].set_title("Silhouette Score theo K", fontsize=13, fontweight="bold")
axes[1].grid(True, alpha=0.3)

best_k_sil = list(K_RANGE)[np.argmax(sil_scores)]
axes[1].axvline(x=best_k_sil, color="red", linestyle="--",label=f"Best K = {best_k_sil}")
axes[1].legend()

plt.suptitle("KMeans — Tìm K tối ưu trên Train Set", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\nK tốt nhất theo Silhouette: K = {best_k_sil} (score = {max(sil_scores):.4f})")
print("\nBảng chi tiết:")
print(f"{'K':>4} | {'WCSS':>12} | {'Silhouette':>10}")
print("-" * 32)
for k, w, s in zip(K_RANGE, wcss, sil_scores):
    print(f"{k:>4} | {w:>12.1f} | {s:>10.4f}")

### Nhận xét Elbow + Silhouette
**Elbow Method (WCSS)**: giảm mạnh đến K≈4–5 rồi phẳng dần về gần 0 — nếu chỉ nhìn biểu đồ này, "khuỷu tay" thực sự nằm ở K=4–5, không phải K=20.

**Silhouette Score**: tăng khá đều theo K nhưng có đợt tụt tại K=5 (0,60, thấp hơn cả K=4); từ K=13 trở đi gần như đi ngang (0,734–0,756). K=20 đạt điểm cao nhất (0,7564) song cũng là biên trên của dải khảo sát, nên chưa chắc là điểm dừng tối ưu tuyệt đối.

In [ ]:
# Fit KMeans với K tối ưu & visualize
# K_BEST lấy từ kết quả Silhouette tốt nhất ở trên
K_BEST = 20
km_best = KMeans(n_clusters=K_BEST, random_state=42, n_init=10)
kmeans_labels_train = km_best.fit_predict(train_coords)

sil_km = silhouette_score(train_coords,kmeans_labels_train, sample_size=min(5000, len(train_coords)),random_state=42)
print(f"KMeans (K={K_BEST}) — Kết quả trên train set:")
print(f"  Silhouette    : {sil_km:.4f}")
print(f"  WCSS          : {km_best.inertia_:,.1f}")

fig, ax = plt.subplots(figsize=(10, 7))
for k, col in enumerate(cm.tab20(np.linspace(0, 1, K_BEST))):
    mask_k = kmeans_labels_train == k
    ax.scatter(df_train["Longitude"][mask_k], df_train["Latitude"][mask_k],c=[col], s=8, label=f"Cluster {k}", alpha=0.6)

# Centroids (từ radians → degrees để vẽ đúng tọa độ)
centroids_deg = np.degrees(km_best.cluster_centers_)
ax.scatter(centroids_deg[:, 1], centroids_deg[:, 0],c="black", s=80, marker="*", zorder=5, label="Centroids")
ax.set_xlabel("Longitude", fontsize=11)
ax.set_ylabel("Latitude", fontsize=11)
ax.set_title(f"KMeans Clustering (K={K_BEST})\nSilhouette = {sil_km:.4f}",fontsize=13, fontweight="bold")
ax.legend(loc="upper left", fontsize=7, ncol=2, markerscale=2)
plt.tight_layout()
plt.show()

### Nhận xét phân cụm không gian
- Bản đồ phân cụm KMeans (K=20, Silhouette = 0,7602 trên tập Train) cho thấy các cụm phân chia khá rõ ràng theo khu vực địa lý, ít chồng lấn — phù hợp để dùng `geo_region_id` như một biến phân loại vị trí (One-Hot Encoding), không nên hiểu là biến số có thứ tự.
- Tập Test chỉ được gán cụm bằng mô hình KMeans đã fit trên Train (dùng `.predict()`), đảm bảo không có rò rỉ dữ liệu.

In [ ]:
# location_rank theo Out-of-Fold Target Encoding
cv_strategy = KFold(n_splits=5,shuffle=True,random_state=42)
cv_splits = list(cv_strategy.split(df_train))

df_train["location_rank"] = np.nan

for fold_id, (fit_idx, valid_idx) in enumerate(cv_splits, start=1):
    fold_fit = df_train.iloc[fit_idx]
    fold_valid = df_train.iloc[valid_idx]

    fold_stats = (fold_fit.groupby("location_area")["log_price"].median().sort_values())

    fold_rank_map = {area: rank
        for rank, area in enumerate(fold_stats.index, start=1)}
    fold_fallback = (np.median(list(fold_rank_map.values()))
        if fold_rank_map else 1.0)

    df_train.loc[valid_idx, "location_rank"] = (
        fold_valid["location_area"]
        .map(fold_rank_map)
        .fillna(fold_fallback)
        .to_numpy())

# Test được map bằng thống kê của toàn bộ Train
full_area_stats = (df_train.groupby("location_area")["log_price"].median().sort_values())
full_rank_map = {area: rank
    for rank, area in enumerate(full_area_stats.index, start=1)}
full_fallback = (np.median(list(full_rank_map.values()))
    if full_rank_map else 1.0)

df_test["location_rank"] = (
    df_test["location_area"]
    .map(full_rank_map)
    .fillna(full_fallback)
    .astype(float))
df_train["location_rank"] = df_train["location_rank"].astype(float)

assert df_train["location_rank"].isna().sum() == 0
assert df_test["location_rank"].isna().sum() == 0

print("location_rank Train được tạo OOF theo đúng 5 fold dùng trong CV.")
print("location_rank Test chỉ dùng mapping từ toàn bộ Train.")

In [ ]:
train_df = df_train.copy()
test_df = df_test.copy()

train_df["geo_region_id"] = kmeans_labels_train
kmeans_labels_test = km_best.predict(test_coords)
test_df["geo_region_id"] = kmeans_labels_test

# Các cột này đã được chuyển thành đặc trưng tương ứng và không đưa trực tiếp vào X
train_df = train_df.drop(columns=["Price", "Area", "Latitude", "Longitude", "lat_is_missing"],errors="ignore")
test_df = test_df.drop(columns=["Price", "Area", "Latitude", "Longitude", "lat_is_missing"],errors="ignore")

assert set(train_df.columns) == set(test_df.columns)
assert "listing_age" in train_df.columns
assert "location_rank" in train_df.columns
assert "district_tier" in train_df.columns
assert "geo_region_id" in train_df.columns

print(f"Train cuối: {train_df.shape}")
print(f"Test cuối : {test_df.shape}")

### Kết luận B1

- Các ngưỡng nghiệp vụ và deduplication được thực hiện trước Train/Test Split.
- Median và centroid chỉ được fit trên Train rồi áp dụng cho Test.
- `district_tier` được xác định theo địa lý cố định, không dựa vào giá.
- `location_rank` được tạo theo Out-of-Fold trên Train để hạn chế target leakage.
- `geo_region_id` được tạo từ phân cụm tọa độ trên Train và được dùng như biến phân loại.
- Toàn bộ đặc trưng trong `LINREG_FEATURES` và `RF_FEATURES` được giữ lại.


## B2. Xây dựng mô hình

Ba mô hình được sử dụng gồm **Linear Regression**, **Random Forest Regressor** và **XGBoost Regressor**. Tất cả mô hình cùng dự đoán một biến mục tiêu là `log_price`; notebook không xây dựng mô hình riêng cho đơn giá/m². Linear Regression được dùng làm mô hình nền dễ diễn giải, trong khi Random Forest và XGBoost được dùng để học các quan hệ phi tuyến và tương tác phức tạp giữa các đặc trưng.


In [ ]:
# Định nghĩa feature sets
LINREG_FEATURES = [
    'log_area', 'location_rank', 'geo_region_id', 'distance_to_center',
    'Bedrooms', 'Bathrooms', 'Floors', 'Alley Width',
    'bedrooms_is_missing', 'floors_is_missing', 'bathrooms_is_missing',
    'direction_is_missing', 'road_type_is_missing', 'position_is_missing',
    'width_is_missing', 'length_is_missing', 'alley_width_is_missing',
    'has_furniture', 'car_alley', 'near_market', 'near_school', 'is_urgent_sale',
    'listing_age',
    'Direction', 'Position', 'Road Type', 'road_width_bin', 'Property Type']

RF_FEATURES = [
    'log_area', 'location_rank', 'district_tier', 'geo_region_id', 'distance_to_center',
    'Bedrooms', 'Bathrooms', 'Floors', 'Width', 'Length', 'Alley Width',
    'bedrooms_is_missing', 'floors_is_missing', 'bathrooms_is_missing',
    'direction_is_missing', 'road_type_is_missing', 'position_is_missing',
    'width_is_missing', 'length_is_missing', 'alley_width_is_missing',
    'has_furniture', 'car_alley', 'near_market', 'near_school', 'is_urgent_sale',
    'listing_age',
    'Direction', 'Position', 'Road Type', 'road_width_bin', 'Property Type']

print(f"LINREG_FEATURES: {len(LINREG_FEATURES)} features")
print(f"RF_FEATURES    : {len(RF_FEATURES)} features")

In [ ]:
# Kiểm tra toàn bộ đặc trưng bắt buộc tồn tại
missing_lr = [c for c in LINREG_FEATURES if c not in train_df.columns]
missing_rf = [c for c in RF_FEATURES if c not in train_df.columns]

assert not missing_lr, f"Thiếu LINREG_FEATURES: {missing_lr}"
assert not missing_rf, f"Thiếu RF_FEATURES: {missing_rf}"

def prepare_X_y(train_df,test_df,feature_list,categorical_cols,target_col="log_price"):
    X_train = train_df[feature_list].copy()
    X_test = test_df[feature_list].copy()
    y_train = train_df[target_col].copy()
    y_test = test_df[target_col].copy()

    X_train = pd.get_dummies(X_train,columns=categorical_cols,drop_first=True,dtype=float)
    X_test = pd.get_dummies(X_test,columns=categorical_cols,drop_first=True,dtype=float)

    X_test = X_test.reindex(columns=X_train.columns,fill_value=0)

    X_train = X_train.apply(pd.to_numeric, errors="coerce")
    X_test = X_test.apply(pd.to_numeric, errors="coerce")

    X_train = X_train.replace([np.inf, -np.inf], np.nan)
    X_test = X_test.replace([np.inf, -np.inf], np.nan)

    assert X_train.isna().sum().sum() == 0
    assert X_test.isna().sum().sum() == 0

    return (X_train.astype(float),X_test.astype(float),y_train,y_test)

# geo_region_id là nhãn cụm nên bắt buộc One-Hot Encoding
CAT_COLS = ["geo_region_id","Direction","Position","Road Type","road_width_bin","Property Type"]
RF_CAT_COLS = CAT_COLS + ["district_tier"]

X_train_lr, X_test_lr, y_train, y_test = prepare_X_y(train_df,test_df,LINREG_FEATURES,
    [c for c in CAT_COLS if c in LINREG_FEATURES])

X_train_rf, X_test_rf, _, _ = prepare_X_y(train_df,test_df,RF_FEATURES,
    [c for c in RF_CAT_COLS if c in RF_FEATURES])

print(f"LR sau OHE: Train {X_train_lr.shape} | Test {X_test_lr.shape}")
print(f"RF/XGB sau OHE: Train {X_train_rf.shape} | Test {X_test_rf.shape}")

In [ ]:
# Scaling cho mô hình Linear Regression
scaler = StandardScaler()

X_train_lr_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_lr),
    columns=X_train_lr.columns,
    index=X_train_lr.index)

X_test_lr_scaled = pd.DataFrame(
    scaler.transform(X_test_lr),
    columns=X_test_lr.columns,
    index=X_test_lr.index)

print("StandardScaler chỉ fit trên Train.")

Nhóm chỉ thực hiện scaling cho mô hình hồi quy vì mô hình này nhạy hơn với thang đo của các biến. Còn Random Forest và XGBoost là mô hình dựa trên cây quyết định, chia dữ liệu theo ngưỡng và thứ tự giá trị nên không phụ thuộc nhiều vào đơn vị hay độ lớn của đặc trưng.

## Mô hình 1 — Linear Regression

Linear Regression được sử dụng làm baseline vì đơn giản, dễ diễn giải và cho phép quan sát chiều tác động của các hệ số sau chuẩn hóa. Mô hình giả định quan hệ giữa đặc trưng và `log_price` gần tuyến tính, vì vậy kết quả được dùng làm mốc so sánh với các mô hình cây phi tuyến.


In [ ]:
# Fit Linear Regression (sklearn)
lr_model = LinearRegression()
lr_model.fit(X_train_lr_scaled, y_train)

y_pred_lr_train = lr_model.predict(X_train_lr_scaled)
y_pred_lr_test = lr_model.predict(X_test_lr_scaled)

print(f"Linear Regression — R² Train: "f"{r2_score(y_train, y_pred_lr_train):.4f} | "f"R² Test: {r2_score(y_test, y_pred_lr_test):.4f}")

## Mô hình 2 — Random Forest Regressor

Random Forest phù hợp với dữ liệu bất động sản vì có thể mô hình hóa quan hệ phi tuyến và tương tác giữa diện tích, vị trí, đặc điểm kết cấu và tiện ích. `RandomizedSearchCV` được dùng để tìm siêu tham số theo **RMSE**, cùng chiến lược 5-fold đã dùng khi tạo `location_rank`.


In [ ]:
# RandomizedSearchCV cho Random Forest (Tuning có chủ đích)
rf_base = RandomForestRegressor(random_state=42,n_jobs=-1)

rf_param_distributions = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [10, 15, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", 0.8, None]}

rf_search = RandomizedSearchCV(
    estimator=rf_base,n_iter=30,n_jobs=-1,
    param_distributions=rf_param_distributions,
    cv=cv_splits,random_state=42,verbose=1,
    scoring="neg_root_mean_squared_error")

rf_search.fit(X_train_rf, y_train)
best_rf = rf_search.best_estimator_

y_pred_rf_train = best_rf.predict(X_train_rf)
y_pred_rf_test = best_rf.predict(X_test_rf)

print("Best Random Forest parameters:", rf_search.best_params_)
print(f"Best CV RMSE: {-rf_search.best_score_:.4f}")
print(f"R² Train: {r2_score(y_train, y_pred_rf_train):.4f} | "f"R² Test: {r2_score(y_test, y_pred_rf_test):.4f}")

Sau khi tối ưu siêu tham số, Random Forest đạt CV RMSE = 0,5232, R² Train = 0,8280 và R² Test = 0,6298. Mô hình giải thích được khoảng 63% biến thiên của dữ liệu kiểm tra, cho thấy khả năng dự đoán khá tốt. Tuy nhiên, chênh lệch R² Train–Test khoảng 0,1982 cho thấy vẫn có dấu hiệu overfitting nhất định.

## Mô hình 3 — XGBoost Regressor

XGBoost xây dựng các cây tuần tự, trong đó mỗi cây mới tập trung sửa phần sai số còn lại. Mô hình được tuning theo cùng chỉ số RMSE và cùng các fold với Random Forest để bảo đảm so sánh nhất quán.


In [ ]:
# XGBoost Regressor với hyperparameter tuning có chủ đích
try:
    from xgboost import XGBRegressor
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "xgboost"])
    from xgboost import XGBRegressor

xgb_base = XGBRegressor(objective="reg:squarederror",random_state=42,n_jobs=-1,tree_method="hist")
xgb_param_distributions = {
    "n_estimators": [300, 500, 700],
    "learning_rate": [0.03, 0.05, 0.08],
    "max_depth": [3, 5, 7],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.75, 0.85, 1.0],
    "colsample_bytree": [0.75, 0.85, 1.0],
    "reg_alpha": [0.0, 0.1, 0.5],
    "reg_lambda": [1.0, 2.0, 5.0]}

xgb_search = RandomizedSearchCV(
    estimator=xgb_base,n_iter=20,n_jobs=-1,
    param_distributions=xgb_param_distributions,
    scoring="neg_root_mean_squared_error",
    cv=cv_splits,random_state=42,verbose=1)

xgb_search.fit(X_train_rf, y_train)
xgb_model = xgb_search.best_estimator_

y_pred_xgb_train = xgb_model.predict(X_train_rf)
y_pred_xgb_test = xgb_model.predict(X_test_rf)

print("Best XGBoost parameters:", xgb_search.best_params_)
print(f"Best CV RMSE: {-xgb_search.best_score_:.4f}")
print(f"R² Train: {r2_score(y_train, y_pred_xgb_train):.4f} | "f"R² Test: {r2_score(y_test, y_pred_xgb_test):.4f}")

XGBoost đạt R² Test = 0,6353, cho thấy mô hình giải thích được khoảng 63,5% biến thiên của dữ liệu kiểm tra. Chênh lệch R² Train–Test khoảng 0,1331, cho thấy mô hình có khả năng tổng quát hóa tương đối tốt và mức overfitting thấp hơn Random Forest.

## B3. Đánh giá mô hình

Đánh giá gồm Cross-Validation 5-fold, MAE, RMSE, MAPE, R² và trực quan hóa sai số.


In [ ]:
# Cross-Validation với cùng 5 fold cho cả ba mô hình
scoring = {"r2": "r2","rmse": "neg_root_mean_squared_error"}

# Pipeline đảm bảo scaler của Linear Regression được fit lại trong từng fold
lr_cv_pipeline = Pipeline([("scaler", StandardScaler()),("model", LinearRegression())])

cv_lr = cross_validate(lr_cv_pipeline,X_train_lr,y_train,cv=cv_splits,scoring=scoring,n_jobs=-1)
cv_rf = cross_validate(best_rf,X_train_rf,y_train,cv=cv_splits,scoring=scoring,n_jobs=-1)
cv_xgb = cross_validate(xgb_model,X_train_rf,y_train,cv=cv_splits,scoring=scoring,n_jobs=-1)
cv_summary = pd.DataFrame({
    "Model": ["Linear Regression","Random Forest","XGBoost"],
    "R² Mean": [cv_lr["test_r2"].mean(),cv_rf["test_r2"].mean(),cv_xgb["test_r2"].mean()],
    "R² Std": [cv_lr["test_r2"].std(),cv_rf["test_r2"].std(),cv_xgb["test_r2"].std()],
    "RMSE Mean": [-cv_lr["test_rmse"].mean(),-cv_rf["test_rmse"].mean(),-cv_xgb["test_rmse"].mean()],
    "RMSE Std": [cv_lr["test_rmse"].std(),cv_rf["test_rmse"].std(),cv_xgb["test_rmse"].std()]})

display(cv_summary.round(4))

Qua Cross Validation, XGBoost cho kết quả tốt nhất vì có R² cao nhất và RMSE thấp nhất. Random Forest đứng thứ hai, còn Linear Regression thấp nhất. Độ lệch chuẩn của cả ba mô hình đều nhỏ nên kết quả tương đối ổn định và không phụ thuộc quá nhiều vào một lần chia dữ liệu.

In [ ]:
def evaluate_regression(y_true_log, y_pred_log, model_name="Model"):
    """Đánh giá trên log scale và trên giá thật, đơn vị triệu VND."""

    mae_log = mean_absolute_error(y_true_log, y_pred_log)
    rmse_log = np.sqrt(mean_squared_error(y_true_log, y_pred_log))
    r2 = r2_score(y_true_log, y_pred_log)

    # Chuyển ngược từ log1p về giá thật
    y_true_price = np.expm1(y_true_log)
    y_pred_price = np.maximum(np.expm1(y_pred_log), 0)

    mae_price = mean_absolute_error(y_true_price, y_pred_price)
    mape_price = mean_absolute_percentage_error(y_true_price,y_pred_price) * 100

    print(f"{model_name} — Test Set")
    print(f"MAE log                    : {mae_log:.4f}")
    print(f"RMSE log                   : {rmse_log:.4f}")
    print(f"R²                         : {r2:.4f}")
    print(f"MAE giá thật               : {mae_price:,.2f} triệu VND")
    print(f"MAPE giá thật              : {mape_price:.2f}%")
    print()

    return {
        "Model": model_name,
        "MAE Log": mae_log,
        "RMSE Log": rmse_log,
        "R2": r2,
        "MAE giá thật (triệu VND)": mae_price,
        "MAPE giá thật (%)": mape_price}

metrics_lr = evaluate_regression(y_test, y_pred_lr_test, "Linear Regression")
metrics_rf = evaluate_regression(y_test, y_pred_rf_test, "Random Forest")
metrics_xgb = evaluate_regression(y_test, y_pred_xgb_test, "XGBoost")

Trên tập Test, XGBoost đạt kết quả tốt nhất với R² cao nhất (0,6353) và các chỉ số sai số thấp nhất. Random Forest có kết quả gần tương đương, trong khi Linear Regression có sai số lớn hơn rõ rệt. Điều này cho thấy các mô hình cây, đặc biệt là XGBoost, mô tả tốt hơn các quan hệ phi tuyến trong dữ liệu giá bất động sản.

## B4. So sánh mô hình


In [ ]:
# Bảng so sánh: metrics Test và Cross-Validation
metrics_df = pd.DataFrame([metrics_lr,metrics_rf,metrics_xgb])

metric_columns = [
    "Model",
    "MAE Log",
    "RMSE Log",
    "R2",
    "MAE giá thật (triệu VND)",
    "MAPE giá thật (%)"]

comparison_df = metrics_df[metric_columns].merge(cv_summary,on="Model",how="left",validate="one_to_one")

# Bảo vệ khỏi cột trùng nếu cell được chỉnh sửa về sau
comparison_df = comparison_df.loc[:,~comparison_df.columns.duplicated()].copy()

assert comparison_df.columns.is_unique, ("Bảng comparison còn tên cột bị trùng.")
comparison_df = comparison_df.sort_values("RMSE Mean").reset_index(drop=True)
comparison_display = comparison_df.rename(columns={"R2": "R² Test", "RMSE Mean": "CV RMSE Mean"})

print("BẢNG SO SÁNH TỔNG HỢP")
display(comparison_display.round(4))

best_model_name = comparison_df.loc[0, "Model"]
print("Mô hình đề xuất theo CV RMSE thấp nhất: "f"{best_model_name}")
print("Test set chỉ được dùng để báo cáo hiệu suất cuối, ""không dùng làm tiêu chí chọn mô hình.")

### Nhận xét so sánh
- Linear Regression là baseline dễ diễn giải nhưng bị giới hạn bởi giả định tuyến tính.
- Random Forest và XGBoost có khả năng học quan hệ phi tuyến và tương tác giữa các đặc trưng.
- Mô hình được đề xuất theo **CV RMSE thấp nhất**; các chỉ số Test được giữ lại để đánh giá cuối cùng.
- MAE và MAPE trên giá thật được tính sau khi đổi `log_price` về đơn vị triệu VND.

In [ ]:
# Phân tích định lượng Random Forest so với Linear Regression
lr_abs_residual = np.abs(y_test - y_pred_lr_test)
rf_abs_residual = np.abs(y_test - y_pred_rf_test)

lr_train_r2 = r2_score(y_train, y_pred_lr_train)
rf_train_r2 = r2_score(y_train, y_pred_rf_train)

rf_vs_lr = pd.DataFrame({
    "Chỉ tiêu": [
        "R² Test",
        "RMSE Test (log_price)",
        "MAE Test (log_price)",
        "MAE giá thật (triệu VND)",
        "Median |Residual|",
        "P90 |Residual|",
        "Khoảng cách R² Train - Test"],
    "Linear Regression": [
        metrics_lr["R2"],
        metrics_lr["RMSE Log"],
        metrics_lr["MAE Log"],
        metrics_lr["MAE giá thật (triệu VND)"],
        np.median(lr_abs_residual),
        np.percentile(lr_abs_residual, 90),
        lr_train_r2 - metrics_lr["R2"]],
    "Random Forest": [
        metrics_rf["R2"],
        metrics_rf["RMSE Log"],
        metrics_rf["MAE Log"],
        metrics_rf["MAE giá thật (triệu VND)"],
        np.median(rf_abs_residual),
        np.percentile(rf_abs_residual, 90),
        rf_train_r2 - metrics_rf["R2"]]})

rf_vs_lr["Chênh lệch RF - LR"] = (rf_vs_lr["Random Forest"]- rf_vs_lr["Linear Regression"])

display(rf_vs_lr.round(4))

r2_gain = metrics_rf["R2"] - metrics_lr["R2"]
rmse_reduction = ((metrics_lr["RMSE Log"] - metrics_rf["RMSE Log"])/ metrics_lr["RMSE Log"] * 100)
mae_price_reduction = ((metrics_lr["MAE giá thật (triệu VND)"]- metrics_rf["MAE giá thật (triệu VND)"])/ metrics_lr["MAE giá thật (triệu VND)"] * 100)

print("KẾT LUẬN ĐỊNH LƯỢNG")
print(f"• Chênh lệch R² Test: {r2_gain:+.4f}")
print(f"• Tỷ lệ giảm RMSE log: {rmse_reduction:.2f}%")
print(f"• Tỷ lệ giảm MAE giá thật: {mae_price_reduction:.2f}%")
print(f"• Generalization gap — RF: "f"{rf_train_r2 - metrics_rf['R2']:.4f}; "f"LR: {lr_train_r2 - metrics_lr['R2']:.4f}")

if (metrics_rf["R2"] > metrics_lr["R2"]and metrics_rf["RMSE Log"] < metrics_lr["RMSE Log"]):
    print("→ Kết quả thực nghiệm ủng hộ nhận định Random Forest ""mô hình hóa tốt hơn quan hệ phi tuyến và tương tác.")
else:
    print("→ Kết quả lần chạy này chưa cho thấy Random Forest ""vượt Linear Regression đồng thời ở R² và RMSE.")

Random Forest vượt Linear Regression với R² Test cao hơn 0,1840, RMSE giảm 18,27% và MAE giảm 16,90%. Các chỉ số phần dư cũng thấp hơn, cho thấy Random Forest mô hình hóa tốt hơn các quan hệ phi tuyến trong dữ liệu.

In [ ]:
# Phân tích phần dư của Linear Regression
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
residuals_lr = y_test - y_pred_lr_test

axes[0].scatter(y_pred_lr_test,residuals_lr,alpha=0.3,s=12,color=ACCENT)
axes[0].axhline(0,color=RED_LINE,linestyle="--",linewidth=1.8)
axes[0].set_xlabel("Giá trị dự đoán (log_price)")
axes[0].set_ylabel("Phần dư")
axes[0].set_title("Residual Plot — Linear Regression",fontweight="bold")

axes[1].hist(residuals_lr,bins=45,color=ACCENT,alpha=0.85,edgecolor="white")
axes[1].axvline(0,color=RED_LINE,linestyle="--",linewidth=1.8)
axes[1].set_xlabel("Phần dư")
axes[1].set_ylabel("Tần suất")
axes[1].set_title("Phân phối phần dư — Linear Regression",fontweight="bold")

plt.suptitle("Phân tích phần dư của Linear Regression",fontweight="bold",y=1.02)
plt.tight_layout()
plt.show()

### Nhận xét phần dư

- Phần dư chủ yếu tập trung quanh 0 nhưng độ phân tán thay đổi theo giá trị dự đoán, cho thấy dấu hiệu phương sai sai số không đồng nhất và mô hình tuyến tính chưa mô tả hết cấu trúc dữ liệu.
- Histogram phần dư tập trung quanh 0 nhưng chưa hoàn toàn đối xứng và có đuôi dài, cho thấy phân phối phần dư chưa hoàn toàn chuẩn.

In [ ]:
# Predicted vs Actual cho cả 3 mô hình
fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))

models_plot = [
    ("Linear Regression", y_pred_lr_test, metrics_lr),
    ("Random Forest", y_pred_rf_test, metrics_rf),
    ("XGBoost", y_pred_xgb_test, metrics_xgb)]

min_val = min(y_test.min(),*(pred.min() for _, pred, _ in models_plot))
max_val = max(y_test.max(),*(pred.max() for _, pred, _ in models_plot))

for ax, (name, pred, metrics) in zip(axes, models_plot):
    ax.scatter(y_test,pred,color=MODEL_COLORS[name],alpha=0.35,s=15)
    ax.plot([min_val, max_val],[min_val, max_val],color=PERFECT_LINE_COLOR,linestyle="--",linewidth=2,label="Perfect Prediction")
    ax.set_xlim(min_val, max_val)
    ax.set_ylim(min_val, max_val)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(f"{name}\n"f"R² = {metrics['R2']:.3f} | "f"RMSE = {metrics['RMSE Log']:.3f}",fontsize=12,fontweight="bold")
    ax.set_xlabel("log_price thực tế")
    ax.set_ylabel("log_price dự đoán")
    ax.grid(alpha=0.3)
    ax.legend()

plt.suptitle("Predicted vs Actual Comparison (Test Set)",fontsize=14,fontweight="bold",y=1.02)
plt.tight_layout()
plt.show()

### Nhận xét Predicted vs Actual

- Random Forest có điểm dự đoán bám sát đường "Perfect Prediction" hơn hẳn Linear Regression (R² = 0,4458, RMSE = 0,631), đặc biệt ở vùng giá thấp và trung bình.
- Cả 3 mô hình đều có xu hướng dự đoán thấp hơn thực tế ở vùng log_price cao (đuôi phân phối) — hệ quả tự nhiên của việc dữ liệu thưa dần ở phân khúc giá rất cao, khiến mô hình khó học đủ pattern ở vùng này.
- XGBoost nhỉnh hơn Random Forest một chút trên tất cả chỉ số, nhưng khoảng cách không lớn — cả hai đều vượt trội rõ rệt so với Linear Regression.

In [ ]:
# So sánh Top 15 Feature quan trọng của 3 mô hình
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# 1. Linear Regression
coef_lr = pd.Series(lr_model.coef_,index=X_train_lr_scaled.columns)
top15_lr = coef_lr.abs().sort_values().tail(15)
top15_lr_signed = coef_lr.loc[top15_lr.index].sort_values()

colors_lr = [GREEN if value > 0 else RED_LINEfor value in top15_lr_signed.values]

axes[0].barh(top15_lr_signed.index,top15_lr_signed.values,color=colors_lr,edgecolor="white",alpha=0.9)
axes[0].axvline(0,color="black",linewidth=1)
axes[0].set_title("Linear Regression\nTop 15 Standardized Coefficients",fontsize=12,fontweight="bold")
axes[0].set_xlabel("Standardized Coefficient (β)")
axes[0].grid(axis="x", alpha=0.3)

# 2. Random Forest
rf_importance = pd.Series(best_rf.feature_importances_,index=X_train_rf.columns).sort_values().tail(15)
axes[1].barh(rf_importance.index,rf_importance.values,color=ORANGE,edgecolor="white",alpha=0.9)
axes[1].set_title("Random Forest\nTop 15 Feature Importances",fontsize=12,fontweight="bold")
axes[1].set_xlabel("Feature Importance")
axes[1].grid(axis="x", alpha=0.3)

# 3. XGBoost
xgb_importance = pd.Series(xgb_model.feature_importances_,index=X_train_rf.columns).sort_values().tail(15)
axes[2].barh(xgb_importance.index,xgb_importance.values,color=ACCENT,edgecolor="white",alpha=0.9)
axes[2].set_title("XGBoost\nTop 15 Feature Importances",fontsize=12,fontweight="bold")
axes[2].set_xlabel("Feature Importance")
axes[2].grid(axis="x", alpha=0.3)

plt.suptitle("So sánh Top 15 Đặc trưng quan trọng của 3 mô hình",fontsize=15,fontweight="bold",y=1.02)
plt.tight_layout()
plt.show()

### Nhận xét Tầm quan trọng của đặc trưng trong 3 mô hình

- Cả Random Forest và XGBoost đều đồng thuận: **log_area** và **distance_to_center** là hai đặc trưng quan trọng nhất, củng cố kết luận "diện tích và vị trí là hai yếu tố ảnh hưởng mạnh nhất đến giá".
- Với Linear Regression hệ số nằm bên phải đường 0 tác động dương(tăng giá), hệ số bên trái có tác động âm(giảm giá).
- Tầm quan trọng của đặc trưng mô hình cây chỉ thể hiện độ quan trọng, không thể hiện trực tiếp chiều tăng hoặc giảm.

## Giải thích XGBoost bằng SHAP

SHAP (SHapley Additive exPlanations) phân rã từng dự đoán thành mức đóng góp của từng đặc trưng so với giá trị dự đoán nền. Phần này bổ sung khả năng giải thích cho XGBoost ở hai mức:

- **Global importance:** xếp hạng đặc trưng theo trung bình giá trị tuyệt đối SHAP.
- **Beeswarm:** thể hiện đồng thời độ lớn và chiều tác động của từng đặc trưng. Điểm nằm bên phải làm tăng dự đoán `log_price`, điểm bên trái làm giảm dự đoán; màu thể hiện giá trị đặc trưng thấp hoặc cao.

SHAP giúp khắc phục hạn chế của Feature Importance truyền thống vì không chỉ cho biết đặc trưng nào quan trọng mà còn thể hiện đặc trưng đó đẩy dự đoán tăng hay giảm.


In [ ]:
# SHAP — Giải thích mô hình XGBoost
try:
    import shap
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "shap"])
    import shap

SHAP_SAMPLE_SIZE = min(1000, len(X_test_rf))
X_shap = X_test_rf.sample(n=SHAP_SAMPLE_SIZE,random_state=42).copy()

# X_train_rf/X_test_rf đã One-Hot Encoding nên toàn bộ là số
X_shap = X_shap.astype(float)

try:
    explainer = shap.TreeExplainer(xgb_model)
    shap_values = explainer(X_shap)
except Exception:
    # Fallback cho một số tổ hợp phiên bản SHAP/XGBoost
    explainer = shap.TreeExplainer(xgb_model.get_booster())
    shap_raw = explainer.shap_values(X_shap)

    if isinstance(shap_raw, list):shap_raw = shap_raw[0]

    expected_value = np.asarray(explainer.expected_value).reshape(-1)[0]

    shap_values = shap.Explanation(
        values=np.asarray(shap_raw),
        base_values=np.repeat(expected_value, len(X_shap)),
        data=X_shap.to_numpy(),
        feature_names=X_shap.columns.tolist())

shap_importance_df = (
    pd.DataFrame({
        "Feature": X_shap.columns,
        "Mean |SHAP|": np.abs(shap_values.values).mean(axis=0)})
    .sort_values("Mean |SHAP|", ascending=False)
    .reset_index(drop=True))

print("TOP 15 ĐẶC TRƯNG QUAN TRỌNG NHẤT THEO SHAP")
display(shap_importance_df.head(15).round(4))

shap.plots.bar(shap_values,max_display=15,show=False)
plt.gcf().set_size_inches(10, 7)
plt.title("XGBoost — SHAP Global Feature Importance",fontsize=13,fontweight="bold")
plt.tight_layout()
plt.show()

shap.plots.beeswarm(shap_values,max_display=15,show=False)
plt.gcf().set_size_inches(11, 7)
plt.title("XGBoost — SHAP Beeswarm",fontsize=13,fontweight="bold")
plt.tight_layout()
plt.show()

### Nhận xét kết quả SHAP
- SHAP Global Importance xác nhận thứ tự quan trọng: **log_area (0,2726) > distance_to_center (0,1671) > listing_age (0,1388) > Floors (0,0964) > Bedrooms (0,0756) > location_rank (0,0438)**.
- SHAP Beeswarm cho thấy log_area có điểm màu đỏ (giá trị cao) tập trung bên phải và điểm xanh (giá trị thấp) tập trung bên trái, diện tích lớn hơn làm tăng giá dự đoán, đúng logic thị trường.



# C. DIỄN GIẢI KẾT QUẢ
## C1. Diễn giải theo ngôn ngữ kinh doanh
Mô hình ước lượng **tổng giá đăng bán tham khảo** dựa trên diện tích, vị trí, loại hình, số phòng và đặc điểm của bất động sản. Đơn giá/m² chỉ được dùng để phân tích và kiểm tra dữ liệu, không phải đầu ra dự đoán của mô hình.

- **MAE giá thật** cho biết trung bình dự đoán lệch bao nhiêu triệu VND so với giá đăng.
- **MAPE giá thật** thể hiện sai số tương đối trung bình theo phần trăm.
- **RMSE trên log_price** nhấn mạnh các trường hợp sai số lớn và phù hợp để so sánh mô hình trên target đã biến đổi log.
- **R²** phản ánh tỷ lệ biến thiên của `log_price` được mô hình giải thích.

Kết quả không thay thế thẩm định chuyên nghiệp, nhưng có thể hỗ trợ sàng lọc tin bất thường và tư vấn giá ban đầu trong phạm vi dữ liệu nghiên cứu.


## C2. Đề xuất hành động kinh doanh
### Ngắn hạn
1. Tích hợp mức giá tham khảo khi người dùng tạo tin đăng và cảnh báo khi giá lệch đáng kể so với mô hình.
2. Ưu tiên kiểm duyệt các tin có Price, Area, đơn giá hoặc số tầng nằm ngoài vùng dữ liệu phổ biến.
### Dài hạn
1. Thu thập giá giao dịch thực tế thay cho chỉ giá đăng bán.
2. Bổ sung dữ liệu pháp lý, mặt tiền, chất lượng nội thất, quy hoạch và khoảng cách đến tiện ích.
3. Huấn luyện lại định kỳ theo thời gian và theo dõi sai số riêng cho từng quận, loại hình và phân khúc giá.


## C3. Giới hạn mô hình và hướng phát triển
### Giới hạn
1. Giá mục tiêu chủ yếu là giá đăng bán, có thể cao hơn giá giao dịch cuối cùng.
2. Dữ liệu còn thiếu một số yếu tố quan trọng như pháp lý, nội thất, mặt tiền và quy hoạch.
3. Mô hình được xây dựng cho TP.HCM và có thể giảm độ chính xác khi áp dụng cho khu vực hoặc thời điểm khác.
### Hướng phát triển
1. Bổ sung dữ liệu giao dịch và dữ liệu không gian từ nguồn đáng tin cậy.
2. Theo dõi mô hình bằng MAE/RMSE theo từng phân khúc để phát hiện suy giảm chất lượng.
3. Thử CatBoost hoặc LightGBM và chỉ thay thế XGBoost khi Cross-Validation và Test set cùng cho kết quả tốt hơn.
4. Triển khai API hoặc giao diện web để hỗ trợ định giá và cảnh báo tin bất thường.


## Tổng kết & Trả lời câu hỏi nghiên cứu
**Q1: Những đặc trưng nào ảnh hưởng mạnh nhất đến giá bất động sản tại TP.HCM?**
**Quy mô (diện tích)** và **Vị trí (location_rank + geo_region_id)** là hai nhóm yếu tố quan trọng nhất. Chúng giải thích phần lớn phương sai giá. Tiếp theo là cấu hình phòng (Bedrooms/Bathrooms/Floors) và các tiện ích/pháp lý trích xuất từ tin đăng (car_alley, has_furniture, near_school).

**Q2: Mô hình dự đoán nào cho kết quả tốt hơn giữa Linear Regression, Random Forest và các mô hình học máy khác?**
**Random Forest (đã tuning)** dự đoán chính xác hơn rõ rệt trên tập Test (R² cao hơn, RMSE/MAE thấp hơn). Lý do: RF bắt được các tương tác phi tuyến giữa diện tích và vị trí, điều mà Linear Regression không nắm bắt được dù đã log-transform và loại đa cộng tuyến. Ngoài ra thì mô hình XGBoost tốt hơn Random Forest.

**Giá trị thực tiễn của dự án:**
Pipeline OSEMN đầy đủ, tuân thủ chống Data Leakage nghiêm ngặt, tạo ra công cụ định giá khách quan có thể hỗ trợ nhà đầu tư, môi giới và người mua nhà ra quyết định tốt hơn so với định giá cảm tính truyền thống.

### Tại sao Random Forest có thể tốt hơn Linear Regression?

Random Forest không chỉ có thể đạt R² cao hơn mà còn phù hợp hơn với cấu trúc của bài toán định giá:

1. **Quan hệ phi tuyến:** tác động của diện tích, khoảng cách đến trung tâm, số tầng và độ rộng hẻm thường thay đổi theo ngưỡng; Random Forest mô tả được các điểm gãy này.
2. **Tương tác giữa đặc trưng:** giá trị của diện tích phụ thuộc vị trí; tác động của hẻm ô tô hoặc số phòng cũng khác nhau giữa các khu vực. Các cây tự học những tương tác này mà không cần khai báo thủ công.
3. **Nhiều phân khúc định giá:** căn hộ, nhà riêng và các nhóm khu vực có quy luật khác nhau. Random Forest học các quy tắc cục bộ thay vì ép toàn bộ dữ liệu vào một phương trình.
4. **Ít nhạy với ngoại lệ và đa cộng tuyến:** trung bình dự đoán từ nhiều cây thường ổn định hơn khi dữ liệu còn các trường hợp cực đoan hoặc các biến tương quan.
5. **Khai thác cờ missing và biến phân loại:** các cờ thiếu dữ liệu, `geo_region_id`, `district_tier` và tiện ích có thể tạo tác động theo ngưỡng, phù hợp với cấu trúc cây.
6. **Đổi lại:** Random Forest khó giải thích trực tiếp bằng một phương trình và có thể overfit; vì vậy cần đối chiếu Test, Cross-Validation và khoảng cách Train–Test.

### Tinh chỉnh siêu tham số
**Random Forest** nhóm chọn 300 cây để tăng độ ổn định, giới hạn độ sâu ở 15 và yêu cầu tối thiểu 5 mẫu để chia, 2 mẫu ở mỗi lá nhằm hạn chế overfit. Mỗi lần chia chỉ xét 80% đặc trưng để tăng tính đa dạng giữa các cây.

**XGBoost** sử dụng 700 cây nhưng learning rate chỉ 0,05, mỗi cây học một phần nhỏ và sửa dần sai số của các cây trước. Độ sâu cây là 5 tầng, dùng 85% dữ liệu mỗi cây dùng 100% các đặc trưng và bổ sung cả L1, L2 regularization để kiểm soát overfit.
